# Atlas Renovável do Nordeste — Modelagem e Validação

Este notebook treina e valida os **modelos de regressão espacial** que preveem o
potencial de geração renovável a partir de coordenadas geográficas, conforme as
Seções 3.3–3.5 do artigo (Documento 12).

- **Features:** `LAT`, `LON` (a altitude é descartada — ver ablação abaixo)
- **Alvos:** `SOLAR_IRRAD` (kWh/m²/dia), `WIND_SPEED` (m/s) e `IP_NE`
- **Modelos:** KNN, Árvore de Decisão, Random Forest, AdaBoost e MLP
- **Validação:** Holdout 80/20, K-Fold (k=10) e Leave-One-Out (LOO)
- **Rastreamento:** MLflow (no pipeline `src/train.py`)

> O pipeline de produção completo, com **logging no MLflow** e salvamento dos
> modelos, está em [`src/train.py`](../src/train.py). Aqui reaproveitamos suas
> funções para demonstrar a mecânica e apresentar os resultados.

In [1]:
import sys
from pathlib import Path

# Torna o pacote `src` importável a partir da raiz do projeto
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from src import train  # carregar_dados, construir_modelos, avaliar, FEATURES, TARGETS

pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("Features:", train.FEATURES)
print("Alvos:   ", list(train.TARGETS))

Features: ['LAT', 'LON']
Alvos:    ['SOLAR_IRRAD', 'WIND_SPEED', 'IP_NE']


## 1. Dados e deduplicação

Algumas estações aparecem mais de uma vez sob o mesmo código WMO (com pequenas
variações de coordenada ou grafia — ex.: *Paulistana*, que surge **duas vezes** no
top-10 do artigo). Mantê-las criaria pontos quase-duplicados que vazam informação
na validação (especialmente no LOO) e inflam *N*. Por isso, `train.carregar_dados()`
agrega por `COD_WMO` antes do treino.

In [2]:
df = train.carregar_dados()
print(f"Estações únicas: {len(df)}  |  Estados: {df['UF'].nunique()}")
df[["UF", "ESTACAO", "LAT", "LON", "ALT",
    "SOLAR_IRRAD_kwh_m2_dia", "WIND_SPEED_ms", "IP_NE"]].head()

2026-06-04 14:22:47,585 [INFO] Dataset: 155 registros → 134 estações únicas (dedup por COD_WMO); features = ['LAT', 'LON'].


Estações únicas: 134  |  Estados: 9


,UF,ESTACAO,LAT,LON,ALT,SOLAR_IRRAD_kwh_m2_dia,WIND_SPEED_ms,IP_NE
0,MA,SAO LUIS,-2.527,-44.214,54.800,4.191,2.312,0.524
1,MA,BALSAS,-7.456,-46.027,271.030,4.657,1.606,0.558
2,MA,CAROLINA,-7.337,-47.460,182.880,4.507,1.790,0.543
3,MA,CHAPADINHA,-3.743,-43.352,104.000,4.769,2.394,0.613
4,MA,TURIACU,-1.661,-45.372,35.860,3.758,2.839,0.505


In [3]:
df[["SOLAR_IRRAD_kwh_m2_dia", "WIND_SPEED_ms", "IP_NE"]].describe().round(3)

,SOLAR_IRRAD_kwh_m2_dia,WIND_SPEED_ms,IP_NE
count,134.000,134.000,134.000
mean,4.612,2.453,0.605
std,0.901,0.929,0.148
min,1.388,0.348,0.184
25%,4.092,1.891,0.517
50%,4.736,2.416,0.602
75%,5.075,2.885,0.690
max,7.516,8.015,1.128


## 2. Seleção de features: por que descartar a altitude (ALT)

Usamos **importância por ablação** (*drop-column*): comparamos o R² (Leave-One-Out,
Random Forest) com `LAT/LON` contra `LAT/LON/ALT`. Incluir a altitude **reduz** o
R² em todos os alvos — ela tem correlação quase nula com os alvos e apenas adiciona
ruído ao espaço de features. Logo, os modelos finais usam apenas `LAT` e `LON`.

In [4]:
dfa = train.analise_ablacao_altitude(df, list(train.TARGETS))
dfa.pivot(index="alvo", columns="features", values="loo_r2")

features,LAT/LON,LAT/LON/ALT
alvo,,
IP_NE,0.076,-0.051
SOLAR_IRRAD,0.031,-0.084
WIND_SPEED,0.163,0.089


## 3. Modelos e grades de hiperparâmetros

Todos os modelos usam um `Pipeline(StandardScaler + estimador)` — a padronização é
essencial para KNN e MLP e inofensiva para os modelos de árvore, garantindo uma
comparação justa. A busca é feita com `GridSearchCV` (KNN, Árvore, Random Forest,
AdaBoost) e `RandomizedSearchCV` (MLP).

In [5]:
for nome, spec in train.construir_modelos().items():
    print(f"{nome:13s} [{spec['search']:6s}]  {spec['grid']}")

KNN           [grid  ]  {'model__n_neighbors': [3, 5, 7, 9, 11, 15], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2]}
DecisionTree  [grid  ]  {'model__max_depth': [3, 5, 8, 12, None], 'model__min_samples_split': [2, 5, 10]}
RandomForest  [grid  ]  {'model__n_estimators': [100, 300, 500], 'model__max_depth': [5, 10, None]}
AdaBoost      [grid  ]  {'model__n_estimators': [50, 100, 200], 'model__learning_rate': [0.01, 0.1, 1.0]}
MLP           [random]  {'model__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)], 'model__activation': ['relu', 'tanh'], 'model__alpha': [0.0001, 0.001, 0.01, 0.1]}


## 4. Estratégias de validação

Três estratégias são aplicadas, justificadas pelo tamanho reduzido do dataset:

| Estratégia | Para que serve |
|---|---|
| **Holdout 80/20** (seed=42) | linha de base rápida |
| **K-Fold (k=10)** | média ± desvio de R²/MAE/RMSE |
| **Leave-One-Out (LOO)** | menor viés para *N* pequeno → **métrica de seleção** |

O R² do K-Fold é calculado por *fold* (~13 amostras cada) e, por isso, tem
variância alta; o LOO agrega as 134 previsões em um único R², sendo mais estável.
A função `train.avaliar()` aplica as três estratégias.

## 5. Demonstração da mecânica (alvo `IP_NE`)

Para deixar o notebook autocontido, rodamos a busca de hiperparâmetros e as três
validações **ao vivo** para o IP-NE. O pipeline completo (3 alvos × 5 modelos, com
MLflow) está em `src/train.py`.

In [6]:
alvo = "IP_NE"
X = df[train.FEATURES].to_numpy()
y = df[train.TARGETS[alvo]].to_numpy()

linhas = []
for nome, spec in train.construir_modelos().items():
    pipe = train.montar_pipeline(spec["estimator"])
    if spec["search"] == "grid":
        busca = GridSearchCV(pipe, spec["grid"], cv=5, scoring="r2", n_jobs=-1)
    else:
        busca = RandomizedSearchCV(pipe, spec["grid"], n_iter=12, cv=5,
                                   scoring="r2", n_jobs=-1, random_state=42)
    busca.fit(X, y)
    met = train.avaliar(busca.best_estimator_, X, y)
    met.pop("_loo_pred")
    linhas.append({"modelo": nome, **{k: round(float(v), 3) for k, v in met.items()}})

res_ipne = pd.DataFrame(linhas).sort_values("loo_r2", ascending=False)
res_ipne[["modelo", "holdout_r2", "kfold_r2_mean", "kfold_r2_std", "loo_r2"]]

,modelo,holdout_r2,kfold_r2_mean,kfold_r2_std,loo_r2
2,RandomForest,-0.115,-0.043,0.131,0.086
3,AdaBoost,-0.154,-0.039,0.167,0.038
0,KNN,-0.066,-0.101,0.211,0.031
1,DecisionTree,-0.394,-0.230,0.314,-0.069
4,MLP,-0.121,-0.326,0.255,-0.083


## 6. Resultados completos (`src/train.py`)

A tabela abaixo carrega o resultado do pipeline completo (3 alvos × 5 modelos),
gravado em `reports/model_comparison.csv`. Se o arquivo não existir, rode antes:

```bash
python src/train.py
```

In [7]:
comp_path = ROOT / "reports" / "model_comparison.csv"
if comp_path.exists():
    comp = pd.read_csv(comp_path)
    for alvo in train.TARGETS:
        sub = comp[comp["alvo"] == alvo].sort_values("loo_r2", ascending=False)
        print(f"\n=== {alvo} (melhor: {sub.iloc[0]['modelo']}, LOO R²={sub.iloc[0]['loo_r2']:.3f}) ===")
        display(sub[["modelo", "holdout_r2", "kfold_r2_mean", "kfold_r2_std",
                     "loo_r2", "loo_mae", "loo_rmse"]].set_index("modelo").round(3))
else:
    print("Rode `python src/train.py` para gerar reports/model_comparison.csv")


=== SOLAR_IRRAD (melhor: KNN, LOO R²=0.062) ===


,holdout_r2,kfold_r2_mean,kfold_r2_std,loo_r2,loo_mae,loo_rmse
modelo,,,,,,
KNN,-0.029,-0.051,0.276,0.062,0.635,0.869
RandomForest,-0.162,-0.093,0.239,0.036,0.650,0.881
AdaBoost,-0.142,-0.068,0.229,0.035,0.655,0.881
MLP,-0.080,-0.313,0.372,-0.121,0.681,0.950
DecisionTree,-0.455,-0.541,0.953,-0.165,0.730,0.968



=== WIND_SPEED (melhor: KNN, LOO R²=0.154) ===


,holdout_r2,kfold_r2_mean,kfold_r2_std,loo_r2,loo_mae,loo_rmse
modelo,,,,,,
KNN,-0.016,-0.025,0.376,0.154,0.586,0.851
RandomForest,-0.212,-0.234,0.639,0.147,0.586,0.854
AdaBoost,-0.126,-0.122,0.320,0.096,0.608,0.880
MLP,-0.001,-0.141,0.351,0.088,0.613,0.884
DecisionTree,-0.776,-0.659,0.584,-0.209,0.688,1.017



=== IP_NE (melhor: RandomForest, LOO R²=0.086) ===


,holdout_r2,kfold_r2_mean,kfold_r2_std,loo_r2,loo_mae,loo_rmse
modelo,,,,,,
RandomForest,-0.115,-0.043,0.130,0.086,0.106,0.141
AdaBoost,-0.154,-0.039,0.166,0.038,0.109,0.145
KNN,-0.066,-0.101,0.211,0.031,0.107,0.145
DecisionTree,-0.394,-0.230,0.314,-0.069,0.114,0.152
MLP,-0.121,-0.326,0.255,-0.083,0.117,0.153


In [8]:
# Figuras geradas pelo pipeline
for fig in ["fig09_comparacao_modelos.png", "fig10_previsto_vs_real.png",
            "fig11_ablacao_altitude.png"]:
    p = ROOT / "reports" / fig
    if p.exists():
        plt.figure(figsize=(11, 5))
        plt.imshow(plt.imread(p)); plt.axis("off"); plt.show()

## 7. Discussão e conclusões

- **Vento é o alvo mais previsível** a partir de coordenadas (maior R² LOO): tem
  estrutura espacial clara (corredores litorâneos vs. interior).
- **Irradiação solar é praticamente não-previsível** por coordenadas em escala
  regional (R² ≈ 0): o semiárido é uniformemente ensolarado, restando pouca
  variância espacial para os modelos aprenderem. Isso contrasta com o R²=0,89 do
  Kriging do artigo original, obtido em **um único estado** com grade densa — um
  cenário de validação bem mais favorável que o LOO inter-regional aqui.
- **IP-NE** fica entre os dois, por combinar as duas variáveis.
- **K-Fold (k=10) × LOO:** o R² por *fold* é instável (folds pequenos) e chega a
  ficar negativo; o LOO é a métrica de seleção mais confiável para *N* = 134.
- **Altitude** foi descartada por reduzir o R² (ablação, Seção 2).

**Reprodução:** `python src/train.py` treina tudo e registra no MLflow; abra a UI
com `mlflow ui` e o dashboard com `streamlit run app/app.py`.